# 06_02 Less repetition: temperature and top-k

*📖 Book: section 5.3*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → [model writes] → MCP tool
> ```
>
> You are here: **model writes**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/homer_model.pt`
>
> **This notebook:** Always taking the most likely letter loops (`και και και`). Choosing at random fixes that, but too much chance gives gibberish. Temperature and top-k set the balance.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to pick the next letter at random, with likely letters winning more often
- What temperature does to the chances, and what top-k does
- How to repeat a random run exactly (the seed)
- Which setting reads most like Greek, judged with real numbers

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **temperature** | a number we divide the scores by before softmax: below 1 it sharpens the chances, above 1 it flattens them | `scores / 0.5` |
| **top-k** | keep only the k best letters and give all the others zero chance, then pick at random among the k | `top_k=3` |
| **decoding** | the rule that turns the chances into one chosen letter (greedy, sampling, …) | greedy decoding |
| **multinomial** | PyTorch's weighted dice: it picks a position, each with its own chance | `torch.multinomial(chances, 1)` |

## 3. The idea

### ❓ Greedy gets stuck. Why would I ever pick a letter that isn't the best?

Think of a weighted dice, or a bag of 100 tokens where 26 say `␣`, 15 say `ι`, 7 say `τ` and so on (the chances after `ἄνδρα` from [06_01](06_01_writing_greek.ipynb)). We draw one token at random. `␣` comes out most often, but not every time. That's **sampling** ([01_01](../01_foundations/01_01_a_language_model_without_a_brain.ipynb)). The text is different each time, and a loop can't trap it, because the same 24 letters don't always give the same next letter.

The thing is, sampling brings two new problems of its own. Both have a dial.

In [ ]:
import torch

from homer_ai.lessons import CHUNK, load_homer_model

# read-only: we only load the saved model, we never save
tokenizer, model = load_homer_model()  # yours, or else the ready-made one

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: how do I draw a letter at random, and get the same draw again?

`torch.multinomial(chances, num_samples=...)` is the weighted dice. A computer's "random" numbers come from a **seed** ([05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb)). Set the same seed first, and you get the same "random" draws. That keeps our notebooks repeatable.

👉 **Predict first:** 10 draws after `ἄνδρα`. Which character will come up most? And with the seed set again, will the draws be the same?

In [ ]:
def show(ch):  # make invisible characters visible
    return {" ": "␣", "\n": "⏎"}.get(ch, ch)


if model is not None:
    ids = tokenizer.encode("ἄνδρα")
    with torch.no_grad():
        scores = model(torch.tensor(ids))[-1]  # the 137 scores after ἄνδρα
    chances = torch.softmax(scores, dim=-1)

    for attempt in (1, 2):
        torch.manual_seed(42)  # the same seed every time
        draws = torch.multinomial(chances, num_samples=10, replacement=True)
        letters = [show(tokenizer.decode([i])) for i in draws.tolist()]
        print(f"attempt {attempt}:", " ".join(letters))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A mix with six `␣`, two `ι` and two `ὶ` (an `ι` with an accent). And it's exactly the same 10 characters in both attempts, thanks to the seed. `replacement=True` means "put the token back in the bag after each draw", so the same character may come up twice.

</details>

### ❓ Step 2: what happens if I just sample from the plain chances?

We extend `write()` of 06_01. It's the same loop, with a dice instead of `argmax`. It gets the two dials, `temperature` and `top_k` (explained in the next steps), and a `seed`. In this step we set the dials to "do nothing" (`1.0` and `None`).

In [ ]:
def write(model, tokenizer, prompt, n_new, temperature=0.8, top_k=5, seed=0):
    torch.manual_seed(seed)  # the same seed gives the same text
    ids = tokenizer.encode(prompt)
    with torch.no_grad():
        for _ in range(n_new):
            scores = model(torch.tensor(ids[-CHUNK:]))[-1]
            scores = scores / temperature  # dial 1 (Step 3)
            if top_k is not None:  # dial 2 (Step 4)
                kth_best = torch.topk(scores, top_k).values[-1]
                scores = scores.masked_fill(scores < kth_best, float("-inf"))
            chances = torch.softmax(scores, dim=-1)
            ids.append(torch.multinomial(chances, num_samples=1).item())  # the dice
    return tokenizer.decode(ids).replace("\n", "⏎")  # show a line break as ⏎


def plain_write(prompt, seed):  # no dials: temperature 1, every letter allowed
    return write(model, tokenizer, prompt, 60, temperature=1.0, top_k=None, seed=seed)


if model is not None:
    print("seed 0:", plain_write("ἄνδρα", 0))
    print("seed 1:", plain_write("ἄνδρα", 1))
    print("seed 0 again:", plain_write("ἄνδρα", 0))
    top10 = torch.topk(chances, 10).values.sum().item()
    print(f"\nthe 10 most likely letters share {top10:.2f} of the chance")
    print(f"the other 127 letters share {1 - top10:.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

No loop this time. The text is new and odd, different for each seed and identical for the same seed. But it's full of strange accented letters and fragments. Why? Look at the last line. The 127 unlikely letters share nearly a quarter of the chance (0.23), so about one letter in four comes from that tail. One unlikely letter spoils the word, and then the model reads its own mistake.

So plain sampling swings from one problem (loops) to the other (nonsense). The dials let us choose somewhere in between.

</details>

### ❓ Step 3: what's temperature, and what does it do to the chances?

Before softmax, **divide every score by a number T**. T = 1 changes nothing. Think of a cautious student against a daydreamer:

```
T below 1  →  the biggest scores gain more: sharper chances, safe picks, more repetition
T above 1  →  the scores come closer together: flatter chances, bold picks, more nonsense
T → 0      →  the same as greedy      (T can never be exactly 0: we can't divide by 0)
```

A tiny example with two made-up scores, `2.0` and `1.0`:

👉 **Predict first:** at T = 0.5, will the first one's chance go up or down from its T = 1 value?

In [ ]:
two = torch.tensor([2.0, 1.0])  # made-up scores for two possible letters
for temperature in (0.5, 1.0, 2.0):
    divided = two / temperature
    chances_two = torch.softmax(divided, dim=-1)
    rounded = [round(c, 2) for c in chances_two.tolist()]
    print(f"T={temperature}: scores {divided.tolist()}  →  chances {rounded}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

T = 0.5 doubles the scores to `[4, 2]`, and the first letter's chance climbs from 0.73 to 0.88. T = 2 halves them to `[1, 0.5]`, and it falls to 0.62. Next, the same on our real scores after `ἄνδρα`. The picture shows the 8 most likely characters at T = 0.5, 1 and 2.

</details>

In [ ]:
if model is not None:
    import matplotlib.pyplot as plt

    from homer_ai.studio_style import (
        ACCENT,
        BAD,
        GOOD,
        INK,
        MUTED,
        studio_axes,
        studio_figure,
    )

    temperatures = [(0.5, GOOD), (1.0, ACCENT), (2.0, BAD)]
    order = torch.topk(chances, 8).indices.tolist()  # the 8 most likely at T=1
    labels = [show(tokenizer.decode([i])) for i in order]

    title = "The same chances after ἄνδρα, at three temperatures"
    fig = studio_figure("Temperature", title, (10, 6))
    ax = studio_axes(fig)
    width = 0.27
    for n, (temperature, colour) in enumerate(temperatures):
        values = torch.softmax(scores / temperature, dim=-1)[order].tolist()
        positions = [i + (n - 1) * width for i in range(8)]
        ax.bar(positions, values, width, color=colour, label=f"T = {temperature}")
        for x, v in zip(positions, values, strict=True):
            ax.text(x, v + 0.006, f"{v:.2f}", ha="center", color=INK,
                    family="monospace", fontsize=7)
    ax.set_xticks(range(8), labels)
    ax.tick_params(axis="x", labelsize=16, labelcolor=INK)
    ax.set_ylabel("chance")
    ax.grid(axis="x", visible=False)
    ax.set_axisbelow(True)  # grid lines behind the bars
    ax.legend(facecolor="none", edgecolor=MUTED, labelcolor=INK)
    plt.show()

    for temperature, _ in temperatures:
        chances_t = torch.softmax(scores / temperature, dim=-1)
        top3 = torch.topk(chances_t, 3).values.sum().item()
        best_chance = chances_t.max().item()
        print(f"T={temperature}: best letter {best_chance:.2f}, "
              f"top 3 together {top3:.2f}, the other 134 together {1 - top3:.2f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

At T = 0.5 (green) the best letter has 0.60 and the long tail shrinks to 0.16. At T = 2 (pink-red) the bars are low and even (best letter 0.10), and the tail holds 0.78 of the chance. Same scores, only divided. That's all temperature is.

</details>

### ❓ Your turn: what does a temperature of 0.25 do?

The table above stops at T = 0.5. Go colder: work out the chances of the same two made-up scores, `two`, at T = 0.25.

👉 **Predict first:** at T = 0.5 the first letter had 0.88. More or less than that now?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: the chances of `two` at a temperature of 0.25
cold_chances = None

In [ ]:
from homer_ai.lessons import check

check(
    "the chances at T = 0.25",
    cold_chances,
    torch.softmax(torch.tensor([8.0, 4.0]), dim=-1),
    coach="Got about [0.56, 0.44]? You multiplied by the temperature. We DIVIDE "
    "the scores by it, before softmax: a small T makes the scores bigger and "
    "the gap between them wider.",
)

<details><summary>💡 Hint</summary>

Inside the loop above: `divided = two / temperature`, then `torch.softmax(divided, dim=-1)`.

</details>

<details><summary>✅ Answer</summary>

```python
cold_chances = torch.softmax(two / 0.25, dim=-1)
```

About `[0.98, 0.02]`. Dividing by 0.25 turns the scores `[2, 1]` into `[8, 4]`, and softmax gives almost everything to the first letter. Go colder still and you're back at greedy.

</details>

### ❓ Step 4: what's top-k?

A second way to cut the tail: **keep only the k best letters**, and give every other letter a score of minus infinity (`float("-inf")`). Softmax turns minus infinity into a chance of exactly 0. It's the same trick as the mask in [03_03](../03_attention/03_03_hiding_the_future.ipynb). The dice then picks among the k letters left.

With k = 3 after `ἄνδρα`, only `␣`, `ι` and `τ` stay:

👉 **Predict first:** their chances must add up to 1 again. Will `␣` get more than 0.26?

In [ ]:
if model is not None:
    k = 3
    kth_best = torch.topk(scores, k).values[-1]  # the score of the 3rd best letter
    # the others get minus infinity
    kept = scores.masked_fill(scores < kth_best, float("-inf"))
    chances_k = torch.softmax(kept, dim=-1)
    print("letter   before  after top-3")
    for i in torch.topk(chances, k).indices.tolist():
        letter = show(tokenizer.decode([i]))
        print(f"{letter:^6}   {chances[i].item():.2f}    {chances_k[i].item():.2f}")
    above_zero = (chances_k > 0).sum().item()
    print("letters with a chance above 0:", above_zero, "of", len(chances_k))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The three keep their order, and share what the other 134 lost, so `␣` goes from 0.26 to about 0.54. Only 3 letters can ever be picked, so a wild letter can't happen. Top-k does have a weak spot: it's the same k for every situation. Sometimes only 1 letter makes sense, sometimes 50.

</details>

### ❓ Your turn: can you do top-2 by hand?

Five made-up scores this time. Keep the 2 best, cut the other three off, and turn what's left into chances.

👉 **Predict first:** the two best scores are 2 and 1, the same as `two` in step 3. What chances will they get?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
five = torch.tensor([2.0, 1.0, 0.5, 0.0, -1.0])  # made-up scores for five letters

# 👉 do: every score below the 2nd best becomes -inf, then softmax
# 💡 hint: torch.topk(five, 2).values[-1] is the score of the 2nd best letter
top2_chances = None

In [ ]:
from homer_ai.lessons import check

check(
    "the chances after top-2",
    top2_chances,
    torch.softmax(torch.tensor([2.0, 1.0]), dim=-1).tolist() + [0.0, 0.0, 0.0],
    coach="Exactly 2 chances should be above 0. If the others are small but not "
    "0, you set their scores to 0. A score of 0 still gets a share. Only -inf "
    "gives a chance of exactly 0, and it has to go in BEFORE softmax.",
)

<details><summary>💡 Hint</summary>

The cell above does top-3 on the real scores, in three lines: `kth_best`, `masked_fill`, `softmax`. Do the same on `five`, with 2.

</details>

<details><summary>✅ Answer</summary>

```python
five = torch.tensor([2.0, 1.0, 0.5, 0.0, -1.0])  # made-up scores for five letters

second_best = torch.topk(five, 2).values[-1]
kept_two = five.masked_fill(five < second_best, float("-inf"))
top2_chances = torch.softmax(kept_two, dim=-1)
```

`[0.73, 0.27, 0, 0, 0]`. Those are the chances `two` had at T = 1: once the other three are at `-inf`, it's as if they were never there.

</details>

### ❓ How do I turn the dials and look at real writing?

We write from `ἄνδρα` with the same seed and several settings. `top_k=1` keeps one letter, so it **is** greedy, which gives us a free check. `⏎` is a line break.

In [ ]:
SETTINGS = [
    ("greedy (top-k 1)", 1.0, 1),
    ("temperature 0.5", 0.5, None),
    ("temperature 1", 1.0, None),
    ("temperature 2", 2.0, None),
    ("top-k 3", 1.0, 3),
    ("top-k 10", 1.0, 10),
    ("temperature 0.8, top-k 5", 0.8, 5),
]
if model is not None:
    for label, temperature, top_k in SETTINGS:
        text = write(model, tokenizer, "ἄνδρα", 60, temperature, top_k, seed=0)
        print(f"{label:25}", text)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Greedy loops (`και και και …`). Temperature 1 and 2 are nonsense, and at 2 the words run together, with capitals in the middle (`Φιοόρόὰ`). Temperature 0.5 and top-k 3 give short words separated by spaces and line breaks, which looks most like a poem. Top-k 10 is somewhere in between. Mind you, that's one seed, and our eyes are a poor judge. The optional section further down measures it.

</details>

### ❓ So which setting reads most like Greek?

**None of them writes real Greek sentences.** Our model has 11,840 weights and has read one short book, so even the best settings give word-sized pieces in no sensible order. A few real words do appear (`τοι`, `δε`, `κεν`, `ἐν`), and line breaks come in the right places now and then. Among our settings, top-k 3 or a low temperature read most like Homer's Greek. Greedy is useless for writing.

```
too greedy ◀── loops ──  [ low temperature, small top-k ]  ── gibberish ──▶ too random
```

So that's the trade-off: too greedy gives loops, too random gives gibberish. Real chat models often use temperature 0.7 to 1 and top-k around 40 or 50, with far bigger models behind them.

### ❓ Do the dials work the same on the GPT of chapter 4?

Yes. Our `write` works for both models (`lessons.py` has a version that reads the right window for each, and we import it as `write_both`). We use the same prompt, seed and dials, with the small model first and the GPT below it. `gpt_tokenizer` is the GPT's own copy of the table. It gives the same numbers.

In [ ]:
from homer_ai.lessons import load_homer_gpt
from homer_ai.lessons import write as write_both  # our `write`, for either model

gpt_tokenizer, gpt = load_homer_gpt()
if model is not None:
    for label, temperature, top_k in SETTINGS:
        small = write_both(model, tokenizer, "ἄνδρα", 60, temperature, top_k, seed=0)
        big = write_both(gpt, gpt_tokenizer, "ἄνδρα", 60, temperature, top_k, seed=0)
        print(f"{label}\n  small: {small}\n  GPT:   {big}\n")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The dials do the same to both. Greedy repeats most, temperature 2 is noise, and the middle settings are the readable ones. But the GPT's middle settings are better. You get line breaks, accents in the right places, and here and there a real Homeric word (`ἀνδρῶν` "of men", `ηὔδα`, `Ἀθήνη`). A better model doesn't remove the trade-off. It moves the sweet spot to a nicer place.

</details>

## Optional: measuring "reads like Greek" with numbers

### ❓ How can I measure "reads like Greek" with numbers?

This section is optional. It only backs up what you just saw. We use two simple measures, on 200 letters of text, averaged over 5 seeds (0 to 4):

- **different words**: how many of the words are different from each other. Low means repetition.
- **real words**: how many of the words appear somewhere in Book 1. Low means gibberish.

One catch: the model often writes `και`, but Book 1 spells it `καὶ`, with an accent. So we compare words without accents. `unicodedata.normalize("NFD", word)` splits `ἄ` into `α` plus separate accent marks, and `unicodedata.combining(c)` finds those marks, so we drop them. (It's a crude test. A short word like `δ` is easy to hit by luck, and a real word in the wrong place is still not Greek. So treat it as a guide.)

*In a real product the best measure is your users' thumbs up or down. I wrote a small [open-source tool](https://github.com/IcodeNet/langfuse-reporter) that pulls those ratings out of Langfuse and joins them to the question and answer they were about.*

👉 **Predict first:** which setting will be best on *different words*? And on *real words*?

In [ ]:
import unicodedata

from homer_ai.train_homer import BOOKS_DIR, load_books


def plain(word):
    word = unicodedata.normalize("NFD", word.lower().strip(",.;·"))
    return "".join(c for c in word if not unicodedata.combining(c)).replace("ʼ", "")


def word_scores(text):
    words = [plain(w) for w in text.split()][:-1]  # drop the last word: may be cut off
    different = len(set(words)) / len(words)
    real = sum(w in real_words for w in words) / len(words)
    return different, real


if model is not None:
    real_words = {plain(w) for w in load_books(BOOKS_DIR)[0].split()}
    print(f"{'setting':25} {'small model':>22}   {'GPT':>20}")
    print(f"{'':25} different   real words   different   real words")
    for label, temperature, top_k in SETTINGS:
        row = []
        for m, t in [(model, tokenizer), (gpt, gpt_tokenizer)]:
            texts = [
                write_both(m, t, "ἄνδρα", 200, temperature, top_k, n) for n in range(5)
            ]
            runs = [word_scores(text) for text in texts]
            row += [sum(r[0] for r in runs) / 5, sum(r[1] for r in runs) / 5]
        cells = "".join(f"{value:>12.0%}" for value in row)
        print(f"{label:25}{cells}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

There's the trade-off, in numbers. Greedy has the most real words (about 59%) but only about 13% different words, so it says the same few words over and over. Temperature 2 has nearly all different words (99%) and almost no real ones (8%). The middle settings (temperature 0.5, top-k 3, and temperature 0.8 with top-k 5) have about 30 to 35% real words with 80 to 90% different words. That's the best of both, with top-k 3 slightly ahead on real words.

**The GPT columns** tell the same story at a higher level. Greedy gets 54% different words (against 13%, because it loops later) and 58% real words. The middle settings get 52 to 59% real words with 85 to 90% different words, which is nearly twice the small model's real words at the same variety. One caution, though. The GPT trained on the first 90% of Book 1, so some of its 'real' words may be memorised. And the test only checks that a word exists in Book 1, not that it fits.

</details>

### ❓ What do the numbers say?

They back up what our eyes told us. Top-k 3 or a low temperature look best. Greedy is accurate word by word but useless for writing, and temperature 2 is pure noise.

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Can I use temperature and top-k together?

Yes, and `write` does both. First it divides by the temperature, then it cuts to the k best. The last row of the table did exactly that (temperature 0.8, top-k 5).

## 5. Recap

- Sampling is a weighted dice (`torch.multinomial`). A seed makes it repeat exactly.
- Temperature divides the scores (below 1 sharper, above 1 flatter). Top-k keeps only the k best letters.
- Too greedy loops, too random gives gibberish. Our tiny model never writes real sentences, only less repetitive pieces.

**You can now:**

- sample a letter with `torch.multinomial`, and repeat a run with a seed
- make the chances sharper or flatter with temperature
- cut the choice down to the k best letters with top-k

**Next:** [06_03: The MCP server](06_03_the_mcp_server.ipynb): letting Claude use the model.